# ISLES'22 - Yerel Olcum Paketi (Parca 8 hazirligi)

Parca 8 (gecikme + optimizasyon) **yerel RTX 2060'ta** kosuyor, Colab'da degil - `PLAN.md`
Bolum 3 karari. Bu notebook yerel makineye indirilecek kucuk paketi uretir.

Ayri tutuluyor cunku `isles_egitim.ipynb` 67 hucre ve bu isin egitimle ilgisi yok:
tek yaptigi Drive'daki ciktilari paketlemek.

| Uretilen | Ne |
|---|---|
| `<model>_cikarim.pt` | Yalniz `model` state_dict + konfig + secilen esik. Egitim checkpoint'i optimizer durumu tasidigi icin ~3 kat buyuk; cikarimda gerekmiyor. |
| `olcum_referansi.json` | Test split'i, on isleme konfigi, secilen esikler ve **vaka basina referans Dice** |
| `dagitim.zip` | Yukaridakilerin tek dosyada hali - Drive'dan indirmesi kolay |

`referans_dice` neden onemli: yerel olcum hattinin ayni Dice'i uretmesi hiz olcumunun
on sarti. `hiz_olcumu.py` her vakada kendi urettigi maskeyi bu referansla karsilastiriyor
ve sapma toleransi asilirsa **duruyor**. Aksi halde "su model su kadar hizli" cumlesi iki
farkli seyi olcer.

In [1]:
import os, json, shutil
from pathlib import Path

ORTAM = 'colab' if os.path.isdir('/content') else 'yerel'


def drive_bagla(nokta='/content/drive'):
    if ORTAM != 'colab':
        return None
    from google.colab import drive
    if os.path.ismount(nokta):
        print('Drive zaten bagli:', nokta)
        return nokta
    try:
        drive.mount(nokta)
    except ValueError as e:
        print('duz mount basarisiz (%s) -> force_remount' % e)
        drive.mount(nokta, force_remount=True)
    return nokta


drive_bagla()

PROJE = '/content/drive/MyDrive/iskemik_inme' if ORTAM == 'colab' else str(Path.cwd())
HAZIR = os.path.join(PROJE, 'hazir')
DEG = os.path.join(PROJE, 'degerlendirme')
DAGITIM = os.path.join(PROJE, 'dagitim')
os.makedirs(DAGITIM, exist_ok=True)

import torch
import pandas as pd

CFG = json.load(open(os.path.join(HAZIR, 'config.json')))
SPLIT = json.load(open(os.path.join(PROJE, 'split.json')))
SECIM = json.load(open(os.path.join(DEG, 'secim.json')))
print('PROJE :', PROJE)
print('secim :', SECIM)
print('test  :', len(SPLIT['test']), 'vaka')

Mounted at /content/drive
PROJE : /content/drive/MyDrive/iskemik_inme
secim : {'unet3d': {'esik': 0.7, 'min_voxel': 0}, 'unet_r34_2d': {'esik': 0.2, 'min_voxel': 0}}
test  : 50 vaka


## Model konfigurasyonlari

`isles_egitim.ipynb`'deki `MODEL_CONFIGS`'in ilgili satirlari. Tekrarlanmis olmasi bir risk:
egitim notebooku degisip burasi degismezse yanlis konfig paketlenir. Buna karsi asagida
**state_dict'in ilk katman sekli konfigle karsilastiriliyor** - ayrisirsa hucre hata verir.

In [2]:
MODEL_CONFIGS = {
    'unet_r34_2d': dict(mimari='unet', encoder='resnet34', k=0, batch=32),
    'unet3d':      dict(mimari='unet3d', encoder=None, k=None, batch=2),
}
ORTAK = dict(amp=True, esik=0.5, flair=False)


def konfig(model_key):
    c = dict(ORTAK)
    c.update(MODEL_CONFIGS[model_key])
    c['model_key'] = model_key
    modalite = 3 if c['flair'] else 2
    c['kanal'] = 2 if c['k'] is None else modalite * (2 * c['k'] + 1)
    return c


def ilk_katman_kanali(sd, mimari):
    '''state_dict'ten girdi kanal sayisini okur - konfigle dogrulamak icin.'''
    if mimari == 'unet3d':
        return sd['e1.0.weight'].shape[1]
    return sd['encoder.conv1.weight'].shape[1]


for k in MODEL_CONFIGS:
    print(k, konfig(k))

unet_r34_2d {'amp': True, 'esik': 0.5, 'flair': False, 'mimari': 'unet', 'encoder': 'resnet34', 'k': 0, 'batch': 32, 'model_key': 'unet_r34_2d', 'kanal': 2}
unet3d {'amp': True, 'esik': 0.5, 'flair': False, 'mimari': 'unet3d', 'encoder': None, 'k': None, 'batch': 2, 'model_key': 'unet3d', 'kanal': 2}


## Paketi uret

In [3]:
paket = []
for mk in SECIM:
    if mk not in MODEL_CONFIGS:
        raise KeyError(mk + ' MODEL_CONFIGS icinde yok - secim.json ile bu notebook ayrismis.')
    kaynak = os.path.join(PROJE, mk, 'checkpoints', 'best.pt')
    d = torch.load(kaynak, map_location='cpu', weights_only=False)
    c = konfig(mk)

    gercek = int(ilk_katman_kanali(d['model'], c['mimari']))
    if gercek != c['kanal']:
        raise RuntimeError('{}: checkpoint {} kanal bekliyor, konfig {} diyor. '
                           'MODEL_CONFIGS egitim notebookuyla ayrismis.'
                           .format(mk, gercek, c['kanal']))

    hedef = os.path.join(DAGITIM, mk + '_cikarim.pt')
    torch.save({'model': d['model'], 'konfig': c, 'epoch': d['epoch'],
                'val_dice': d['best_dice'], 'secim': SECIM[mk]}, hedef)
    paket.append(dict(model=mk,
                      kaynak_mb=round(os.path.getsize(kaynak) / 1e6, 1),
                      cikarim_mb=round(os.path.getsize(hedef) / 1e6, 1),
                      kanal=gercek, esik=SECIM[mk]['esik'],
                      min_voxel=SECIM[mk]['min_voxel'],
                      val_dice=round(float(d['best_dice']), 4)))

ref = {}
for mk in SECIM:
    t = pd.read_excel(os.path.join(DEG, mk + '_test_secimli.xlsx'))
    ref[mk] = {r.vaka: float(r.dice) for _, r in t.iterrows()}

json.dump({'split_test': SPLIT['test'], 'cfg': CFG, 'secim': SECIM, 'referans_dice': ref},
          open(os.path.join(DAGITIM, 'olcum_referansi.json'), 'w'), indent=2)

print(pd.DataFrame(paket).to_string(index=False))
print()
print('kanal sayilari checkpointle dogrulandi')

      model  kaynak_mb  cikarim_mb  kanal  esik  min_voxel  val_dice
     unet3d       16.9         5.6      2   0.7          0    0.7081
unet_r34_2d      293.5        97.9      2   0.2          0    0.6777

kanal sayilari checkpointle dogrulandi


In [4]:
zip_yol = shutil.make_archive(os.path.join(PROJE, 'dagitim'), 'zip', DAGITIM)
print('ZIP HAZIR:', zip_yol, round(os.path.getsize(zip_yol) / 1e6, 1), 'MB')
print()
for q in sorted(os.listdir(DAGITIM)):
    print('  ', q, round(os.path.getsize(os.path.join(DAGITIM, q)) / 1e6, 1), 'MB')
print()
print('INDIRME:')
print('  1) drive.google.com -> Drive"im -> iskemik_inme')
print('  2) dagitim.zip dosyasina sag tik -> Indir')
print('  3) Zipi ac, icindeki 3 dosyayi d:/mamografi/iskemik_inme/dagitim/ altina koy')
print('  4) Yerelde: python hiz_olcumu.py --onisleme her-ikisi')

ZIP HAZIR: /content/drive/MyDrive/iskemik_inme/dagitim.zip 96.0 MB

   olcum_referansi.json 0.0 MB
   unet3d_cikarim.pt 5.6 MB
   unet_r34_2d_cikarim.pt 97.9 MB

INDIRME:
  1) drive.google.com -> Drive"im -> iskemik_inme
  2) dagitim.zip dosyasina sag tik -> Indir
  3) Zipi ac, icindeki 3 dosyayi d:/mamografi/iskemik_inme/dagitim/ altina koy
  4) Yerelde: python hiz_olcumu.py --onisleme her-ikisi
